In [1]:
!pip install neo4j

In [4]:
import csv
import os

import_folder_path = r"C:\Users\rcmin\.Neo4jDesktop2\Data\dbmss\dbms-75039c9c-09b4-45da-a11c-42a7b7a9f13b\import" 

for filename in os.listdir(import_folder_path):
    if filename.endswith(".csv"):
        filepath = os.path.join(import_folder_path, filename)
        with open(filepath, 'r', encoding='utf-8-sig') as f: 
            reader = csv.reader(f)
            headers = next(reader)
            print(f"{filename} headers: {headers}")

artist.csv headers: ['artist_uri', 'artist_followers', 'artist_popularity', 'artist_name']
created.csv headers: ['playlist_uri', 'user']
genre.csv headers: ['artist_genres']
in.csv headers: ['song_uri', 'playlist_uris']
labelled.csv headers: ['artist_uri', 'artist_genres']
of_type.csv headers: ['playlist_uri', 'type']
playlist.csv headers: ['playlist_uri', 'name', 'playlist_followers', 'n_tracks']
sing.csv headers: ['song_uri', 'artists_uris']
song.csv headers: ['song_uri', 'name', 'popularity', 'release_date']
type.csv headers: ['type']
user.csv headers: ['user']


In [5]:
from neo4j import GraphDatabase

class SpotifyGraphSetup:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def create_schema(self):
        """Cleans up old constraints by name and creates new ones using exact property names."""
        with self.driver.session() as session:
            # 1. Fetch and drop all existing constraints dynamically
            result = session.run("SHOW CONSTRAINTS YIELD name")
            constraint_names = [record["name"] for record in result]
            
            if constraint_names:
                print("Cleaning up old constraints...")
                for name in constraint_names:
                    session.run(f"DROP CONSTRAINT {name}")
                    print(f" - Dropped: {name}")

            # 2. Create the correct constraints mapped to the CSV headers
            queries = [
                "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Song) REQUIRE s.song_uri IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (a:Artist) REQUIRE a.artist_uri IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Playlist) REQUIRE p.playlist_uri IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (g:Genre) REQUIRE g.name IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Type) REQUIRE t.name IS UNIQUE"
            ]
            
            for query in queries:
                session.run(query)
                
        print("Schema constraints created successfully.")

    def load_data(self):
        """Reads the CSV files from the import folder and builds the graph using batched transactions."""
        
        # 1. Load Nodes (Batched to prevent memory spikes)
        node_queries = [
            """LOAD CSV WITH HEADERS FROM 'file:///song.csv' AS row
               CALL {
                   WITH row
                   MERGE (s:Song {song_uri: row.song_uri})
                   SET s.title = row.name, 
                       s.popularity = toInteger(row.popularity), 
                       s.release_date = row.release_date
               } IN TRANSACTIONS OF 10000 ROWS""", 
               
            """LOAD CSV WITH HEADERS FROM 'file:///artist.csv' AS row
               CALL {
                   WITH row
                   MERGE (a:Artist {artist_uri: row.artist_uri})
                   SET a.name = row.artist_name, 
                       a.followers = toInteger(row.artist_followers), 
                       a.popularity = toInteger(row.artist_popularity)
               } IN TRANSACTIONS OF 10000 ROWS""",
               
            """LOAD CSV WITH HEADERS FROM 'file:///playlist.csv' AS row
               CALL {
                   WITH row
                   MERGE (p:Playlist {playlist_uri: row.playlist_uri})
                   SET p.name = row.name, 
                       p.followers = toInteger(row.playlist_followers), 
                       p.n_tracks = toInteger(row.n_tracks)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///user.csv' AS row
               CALL {
                   WITH row
                   MERGE (u:User {user_id: row.user})
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///genre.csv' AS row
               CALL {
                   WITH row
                   MERGE (g:Genre {name: row.artist_genres})
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///type.csv' AS row
               CALL {
                   WITH row
                   MERGE (t:Type {name: row.type})
               } IN TRANSACTIONS OF 10000 ROWS"""
        ]

        # 2. Load Relationships (Batched to prevent OutOfMemory errors)
        relationship_queries = [
            """LOAD CSV WITH HEADERS FROM 'file:///sing.csv' AS row
               CALL {
                   WITH row
                   MATCH (a:Artist {artist_uri: row.artists_uris}), (s:Song {song_uri: row.song_uri})
                   MERGE (a)-[:SING]->(s)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///created.csv' AS row
               CALL {
                   WITH row
                   MATCH (u:User {user_id: row.user}), (p:Playlist {playlist_uri: row.playlist_uri})
                   MERGE (u)-[:CREATED]->(p)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///in.csv' AS row
               CALL {
                   WITH row
                   MATCH (s:Song {song_uri: row.song_uri}), (p:Playlist {playlist_uri: row.playlist_uris})
                   MERGE (s)-[:IN]->(p)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///of_type.csv' AS row
               CALL {
                   WITH row
                   MATCH (p:Playlist {playlist_uri: row.playlist_uri}), (t:Type {name: row.type})
                   MERGE (p)-[:OFTYPE]->(t)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///labelled.csv' AS row
               CALL {
                   WITH row
                   MATCH (a:Artist {artist_uri: row.artist_uri}), (g:Genre {name: row.artist_genres})
                   MERGE (a)-[:LABELLED]->(g)
               } IN TRANSACTIONS OF 10000 ROWS"""
        ]

        with self.driver.session() as session:
            print("Loading nodes in batches...")
            for query in node_queries:
                session.run(query)
            
            print("Loading relationships in batches (this might take a moment)...")
            for query in relationship_queries:
                session.run(query)
                
        print("Graph data loaded successfully!")

    def get_collaborator_recommendations(self, target_artist):
        """Finds top 10 recommended collaborators for a given artist."""
        query = """
        MATCH (target:Artist {name: $artist_name})
        MATCH (target)-[:SING]->(:Song)-[:IN]->(p:Playlist)<-[:IN]-(:Song)<-[:SING]-(other:Artist)
        WHERE target <> other
        AND other.name <> 'Various Artists'
        AND NOT EXISTS {
            (target)-[:SING]->(:Song)<-[:SING]-(other)
        }
        RETURN other.name AS Potential_Collaborator, count(DISTINCT p) AS Shared_Playlists
        ORDER BY Shared_Playlists DESC
        LIMIT 10
        """
        with self.driver.session() as session:
            result = session.run(query, artist_name=target_artist)
            print(f"\nTop 10 Collaborator Recommendations for {target_artist}:")
            print("-" * 60)
            records = list(result)
            if not records:
                print("No recommendations found or artist not in database.")
                return
            for index, record in enumerate(records, start=1):
                collaborator = record['Potential_Collaborator']
                shared_count = record['Shared_Playlists']
                print(f"{index}. {collaborator} (Shared Playlists: {shared_count})")

In [3]:
URI = "bolt://localhost:7687"
USER = "neo4j" 
PASSWORD = "password"

setup = SpotifyGraphSetup(URI, USER, PASSWORD)
setup.create_schema()
setup.load_data() 
setup.close()

Cleaning up old constraints...
 - Dropped: constraint_5ac005f2
 - Dropped: constraint_60d3be4d
 - Dropped: constraint_bb4804ff
 - Dropped: constraint_d7f52273
 - Dropped: constraint_f4d47ccd
 - Dropped: constraint_f6ec32a1
Schema constraints created successfully.
Loading nodes in batches...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (row) { ... }', position=<SummaryInputPosition line=2, column=16, offset=68>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 68, 'line': 2, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "LOAD CSV WITH HEADERS FROM 'file:///song.csv' AS row\n               CALL {\n                   WITH row\n                   MERGE (s:Song {song_uri: row.song_uri})\n                   SET s.title = row.name, \n                       s.popularity = toInteger(row.popularity), \n                       s.release_date = row.release_date\n    

Loading relationships in batches (this might take a moment)...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (row) { ... }', position=<SummaryInputPosition line=2, column=16, offset=68>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 68, 'line': 2, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "LOAD CSV WITH HEADERS FROM 'file:///sing.csv' AS row\n               CALL {\n                   WITH row\n                   MATCH (a:Artist {artist_uri: row.artists_uris}), (s:Song {song_uri: row.song_uri})\n                   MERGE (a)-[:SING]->(s)\n               } IN TRANSACTIONS OF 10000 ROWS"
Received notification from DBMS server: 

Graph data loaded successfully!


In [10]:
URI = "bolt://localhost:7687"
USER = "neo4j" 
PASSWORD = "password"

setup = SpotifyGraphSetup(URI, USER, PASSWORD)

setup.get_collaborator_recommendations("Porter Robinson")
setup.get_collaborator_recommendations("Madeon")

setup.close()


Top 10 Collaborator Recommendations for Porter Robinson:
------------------------------------------------------------
1. The Chainsmokers (Shared Playlists: 10)
2. ILLENIUM (Shared Playlists: 9)
3. Marshmello (Shared Playlists: 7)
4. David Guetta (Shared Playlists: 6)
5. Avicii (Shared Playlists: 6)
6. Bebe Rexha (Shared Playlists: 6)
7. Said The Sky (Shared Playlists: 5)
8. Zedd (Shared Playlists: 5)
9. ODESZA (Shared Playlists: 5)
10. SLANDER (Shared Playlists: 5)

Top 10 Collaborator Recommendations for Madeon:
------------------------------------------------------------
1. ILLENIUM (Shared Playlists: 7)
2. The Chainsmokers (Shared Playlists: 7)
3. Avicii (Shared Playlists: 6)
4. Flume (Shared Playlists: 5)
5. Ellie Goulding (Shared Playlists: 5)
6. San Holo (Shared Playlists: 5)
7. Marshmello (Shared Playlists: 5)
8. Dua Lipa (Shared Playlists: 5)
9. Kygo (Shared Playlists: 4)
10. Jon Bellion (Shared Playlists: 4)
